In [ ]:
def calculate_all_epsilon(model, class_num, logit_example, step_eps=0.01, max_eps=10.0):
    first_changed_list = []
    targeted_list = []
    for i in range(0, class_num):
        print(f"class {i}")
        targeted_temp = []
        first_changed_temp = []
        for j in range(0, class_num):
            if i == j:
                targeted_temp.append((i, j, math.inf))
                first_changed_temp.append((i, j, math.inf))
                continue
            eps, first_diff_eps = fgsm_attack(model, i, j, logit_example[i], 0.00, step_eps, max_eps, device)
            if eps >= max_eps:
                targeted_temp.append((i, j, math.inf))
            else:
                targeted_temp.append((i, j, eps))
            first_changed_temp.append((i, j, first_diff_eps))
        targeted_list.append(targeted_temp)
        first_changed_list.append(first_changed_temp)
        
    flat_targeted_list = [(source, target, eps) for sublist in targeted_list for source, target, eps in sublist if eps != math.inf]
    flat_first_changed_list = [(source, target, eps) for sublist in first_changed_list for source, target, eps in sublist if eps != math.inf and eps - step_eps >=0.00]
        
    return flat_targeted_list, flat_first_changed_list

In [ ]:
def make_example(model, data_frame, data_loader, tokenizer, example_num, top_emb, bottom_emb, class_num, device, step_eps=0.01, max_eps=10.0):
    extract_embed_list = []
    negative_eps = []
    positive_eps = []
    example_list = []
    example_label = []
    # Positive, Negative 예제 개수 계산
    extract_num = top_emb + bottom_emb

    print("Generate num : ", example_num)
    
    per_emb_example_num = int(round(example_num / extract_num))

    print("Extract Num : ", extract_num)
    print("Generate embedding positive example num : ", per_emb_example_num)

    input_embeds = select_true_example(model, class_num, data_frame, data_loader, tokenizer, device)
    
    extract_embed_list.extend(extract_top_n_embeddings(top_emb, class_num, input_embeds))
    extract_embed_list.extend(extract_bottom_n_embeddings(bottom_emb, class_num, input_embeds))

    for i in range(0, len(extract_embed_list)):
        targeted_eps, first_changed_eps = calculate_all_epsilon(model, class_num, extract_embed_list[i], step_eps, max_eps)
        positive_eps.append(first_changed_eps)
        negative_eps.append(targeted_eps)
        
    for i, input_embed in enumerate(extract_embed_list):
        for j, first_changed in enumerate(positive_eps[i]):
            per_example_num = int(round(per_emb_example_num / len(positive_eps[i])))
            source, target, eps = first_changed
            print("per example num", per_example_num)
            pos_label, pos_example = generate_example(model, device, source, target, extract_embed_list[i], per_example_num, eps-step_eps, step_eps)
            example_label.extend(pos_label)
            example_list.extend(pos_example)
    adjust_examples(example_list, example_label, positive_eps, extract_embed_list, example_num, step_eps, generate_example, model, device)

    return example_label, example_list

In [ ]:
example_label, example_list = make_example(model, data_frame = validation_df, data_loader = validation_loader, tokenizer = tokenizer, example_num=3000 ,top_emb=2, bottom_emb=2, class_num=10, device=device, step_eps=0.01, max_eps=10.0)